# TAA — Taxa de Acerto da Alocação Partidária

## Contexto conceitual

O capítulo 3 mede a coordenação intrapartidária por indicadores que descrevem
**como o partido distribuiu** os recursos: `NECr` (concentração), `Competitivo`
(quem é forte *ex-ante*) e `CorrespondRec = NECr/Mp` (concentração relativa à
referência de cadeiras). Nenhum deles confronta a alocação com o **desfecho**.

A TAA responde à pergunta que falta: *o partido concentrou nos candidatos certos?*

Para cada lista (partido × UF × ano), ordenam-se os candidatos pelos recursos
partidários recebidos, tomam-se os $k = \min(Mp, n^{+})$ primeiros — o *conjunto
designado* — e mede-se a fração deles que se elegeu:

$$\mathrm{TAA}_\ell = \frac{|D_\ell \cap \text{Eleitos}_\ell|}{k_\ell}$$

$Mp$ é a bancada do partido na UF em exercício na véspera das convenções (API da
Câmara). **Ranking e denominador são integralmente ex-ante**; o resultado
eleitoral entra apenas como desfecho avaliado, nunca como insumo da definição —
o que preserva a salvaguarda antitautológica do capítulo (Bolognesi et al. 2020).

Isso distingue a TAA de duas medidas próximas:

- de `share_top_Mp` (teste discriminante de Fiva), criticada por circularidade:
  ordenar por recursos e medir a *fração de recursos* dos top-$Mp$ é tautológico.
  Aqui o numerador (`eleito`) é externo ao critério de ordenação;
- de Cheibub, Junqueira & Moreira (2024), cuja medida de coordenação é
  construída a partir do resultado da própria eleição que serve de VD.

## Benchmark

Uma TAA bruta não é interpretável sozinha: uma lista com $Mp=1$ e 20 candidatos
enfrenta um acaso muito diferente de uma com $Mp=10$ e 30. O benchmark aleatório
é $b_\ell = S_\ell/n_\ell$, e a versão normalizada (*skill score*) é

$$\mathrm{TAA}^{aj}_\ell = \frac{\mathrm{TAA}_\ell - b_\ell}{1 - b_\ell}$$

0 = acaso, 1 = acerto perfeito, negativo = pior que o acaso.

## Ressalvas

- Nomear como "taxa de acerto dos $Mp$ maiores recebedores", **nunca** como
  "taxa de acerto dos candidatos que o partido esperava eleger".
- A TAA não discrimina coordenação partidária de captura individual, nem
  alocação *top-down* de ratificação *bottom-up* (Hoyler & Marques 2023).


In [1]:
import os
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)
import sys
sys.path.insert(0, str(ROOT / 'src' / '2_gold'))

In [2]:
import numpy as np
import pandas as pd

from cap3_taa_features import (
    construir_taa,
    tabela_bateria,
    diagnostico_merge,
    acertos_fracionarios,
    carregar_rrd,
    carregar_bancada,
)

pd.set_option('display.width', 140)

rrd = carregar_rrd()
df = construir_taa(rrd)

print(f"{len(df):,} listas | anos: {sorted(df['ano_eleicao'].unique())}")
df.groupby('ano_eleicao').size().to_frame('n_listas')

770 listas | anos: [2014, 2018, 2022]


,n_listas
ano_eleicao,
2014,230
2018,290
2022,250


## 1. Cobertura do merge do $Mp$

O `rrd` e o snapshot da Câmara usam grafias diferentes das legendas. A
harmonização é ASCII + quatro aliases (`PCDOB`, `PP**`, `SD`, `PTDOB`) +
`groupby.sum()` — este último para que fusões já consolidadas na data do
snapshot (DEM+PSL → UNIÃO em 2022) entrem como uma bancada só.

Deliberadamente **não** se aplicam `DEM→UNIÃO`, `PR→PL` e `PRB→REPUBLICANOS`:
o snapshot já traz a sigla contemporânea de cada ano, e aplicá-los quebraria o
merge.

In [3]:
orfaos = diagnostico_merge(rrd)
total = carregar_bancada()['Mp'].sum()
print(f"cobertura: {100 * (1 - orfaos['Mp'].sum() / total):.2f}%  "
      f"({orfaos['Mp'].sum()} de {total} deputados sem lista correspondente)")
orfaos

cobertura: 99.87%  (2 de 1519 deputados sem lista correspondente)


,ano_eleicao,sg_uf,sg_partido_norm,Mp
248,2014,TO,PDT,1
657,2022,MT,PP,1


Os dois órfãos são casos reais: partido com deputado em exercício na UF que
não lançou lista de federal ali naquela eleição. Não há perda por grafia.

## 2. Listas degeneradas

Listas em que **todos** os candidatos se elegeram ($b_\ell = 1$) têm TAA = 1 por
construção e *skill score* indefinido. São listas mínimas — quase todas com um
único candidato — e sua frequência despenca ao longo do período, porque
dependiam das coligações.

Consequência prática: **comparações entre anos na TAA bruta precisam excluí-las**,
sob pena de atribuir ao FEFC uma queda que é composição de amostra. A TAA
ajustada já as descarta.

In [4]:
deg = df['lista_degenerada']
print("Listas degeneradas por ano:")
display(df.groupby('ano_eleicao')['lista_degenerada'].agg(['sum', 'mean']).round(3))

print("\nTamanho dessas listas:")
display(df[deg]['n_cands'].value_counts().sort_index().to_frame('n_listas'))

print("\nTAA bruta: universo completo vs. universo comparável")
pd.DataFrame({
    'completo': df.groupby('ano_eleicao').taa.mean(),
    'sem_degeneradas': df[~deg].groupby('ano_eleicao').taa.mean(),
    'taa_aj': df.groupby('ano_eleicao').taa_aj.mean(),
}).round(3)

Listas degeneradas por ano:


,sum,mean
ano_eleicao,,
2014,22,0.096
2018,24,0.083
2022,0,0.000



Tamanho dessas listas:


,n_listas
n_cands,
1,40
2,5
3,1



TAA bruta: universo completo vs. universo comparável


,completo,sem_degeneradas,taa_aj
ano_eleicao,,,
2014,0.618,0.578,0.509
2018,0.550,0.509,0.452
2022,0.497,0.497,0.464


A queda bruta 2014→2022 encolhe de 0,618 → 0,497 para 0,578 → 0,497 quando o
universo fica comparável. E a TAA **ajustada** é praticamente estável nos três
ciclos: a queda restante da TAA bruta é artefato do crescimento do pool de
candidatos (o denominador do acaso), não piora da mira do partido.

Esta é a leitura substantiva central — e ela se inverte se o benchmark for
ignorado.

## 3. Resultado principal e bateria comparativa

O mesmo $k_\ell$, trocando apenas o critério de ordenação. É o que converte um
número descritivo em afirmação comparativa: *o dinheiro do partido ordena
vencedores melhor que sinais alternativos?*

| coluna | ranking | pergunta |
|---|---|---|
| `taa` | recursos do partido (desc) | **principal** |
| `taa_outros` | recursos de outras fontes (desc) | bate o dinheiro não-partidário? |
| `taa_lag` | `prop_votos_nominais_lag` (desc) | bate o desempenho passado? |
| `taa_timing` | `dias_desde_inicio` (asc) | antecipação seleciona tão bem quanto montante? |
| `taa_exante` | incumbência × 1000 + voto t−1 (desc) | bate o ranking ex-ante puro? |
| `base` | — | benchmark aleatório $S/n$ |

In [5]:
print("Universo completo:")
display(tabela_bateria(df))

print("\nUniverso comparável (sem listas degeneradas):")
tabela_bateria(df[~deg])

Universo completo:


,taa,taa_outros,taa_lag,taa_timing,taa_exante,base,taa_aj,n_listas
ano_eleicao,,,,,,,,
2014,0.618,0.645,0.494,0.482,0.563,0.282,0.509,230
2018,0.550,0.479,0.345,0.441,0.495,0.240,0.452,290
2022,0.497,0.399,0.402,0.296,0.482,0.115,0.464,250



Universo comparável (sem listas degeneradas):


,taa,taa_outros,taa_lag,taa_timing,taa_exante,base,taa_aj,n_listas
ano_eleicao,,,,,,,,
2014,0.578,0.607,0.440,0.429,0.516,0.206,0.509,208
2018,0.509,0.432,0.286,0.390,0.450,0.171,0.452,266
2022,0.497,0.399,0.402,0.296,0.482,0.115,0.464,250


**O achado central está na comparação `taa` vs. `taa_outros`:**

- **2014 (pré-FEFC)**: o dinheiro do partido *não* seleciona melhor que o dinheiro
  de outras fontes — na verdade seleciona pior. Sem o fundo, o repasse partidário
  não carrega informação distintiva sobre quem vai se eleger.
- **2018 e 2022 (com FEFC)**: a ordem se inverte e a vantagem é substancial.

Ou seja: o FEFC não apenas aumentou o volume sob controle das lideranças — ele
criou um instrumento de alocação que *discrimina*. Um teste de placebo embutido,
com 2014 como linha de base.

Dois resultados secundários: a TAA por **timing** é consistentemente menor que
por montante (a antecipação é sinal mais fraco que o volume — coerente com o
cap. 4), e a TAA do partido supera o **ranking ex-ante puro** em 2018/2022, o
que indica que a liderança usa informação além de incumbência e voto passado.

## 4. Desagregação

In [6]:
tabela_bateria(df[~deg], ['tipo_partido_exante'])[['taa', 'taa_aj', 'base', 'n_listas']]

taa  taa_aj   base  n_listas
ano_eleicao tipo_partido_exante                                
2014        Competitivo          0.625   0.555  0.231       142
            Menos competitivo    0.475   0.410  0.152        66
2018        Competitivo          0.529   0.448  0.209       167
            Menos competitivo    0.476   0.459  0.108        99
2022        Competitivo          0.525   0.484  0.133       176
            Menos competitivo    0.430   0.416  0.071        74

`tipo_partido_exante` usa a bancada nacional **prévia** (≥ 20 cadeiras), não as
cadeiras eleitas na própria eleição. A versão de `gerar_features` é ex-post e
classificaria o PSL de 2018 como competitivo, contaminando a leitura.

Partidos competitivos acertam mais em todos os ciclos — o gradiente previsto por
Fiva et al. (2024) para coordenação-*gatekeeping*, aqui com uma VD que **não**
sofre do artefato de tamanho de lista que afeta `share_exante_Mp` (o denominador
da TAA é $k \le Mp$, fixo, não o tamanho da lista).

In [7]:
tabela_bateria(df[~deg], ['dm_cat'])[['taa', 'taa_aj', 'base', 'n_listas']]

taa  taa_aj   base  n_listas
ano_eleicao dm_cat                                        
2014        Grande (39–70)  0.564   0.479  0.172        64
            Médio (16–31)   0.642   0.579  0.240        69
            Pequeno (8–12)  0.530   0.471  0.204        75
2018        Grande (39–70)  0.561   0.485  0.147        69
            Médio (16–31)   0.529   0.473  0.172        98
            Pequeno (8–12)  0.453   0.409  0.187        99
2022        Grande (39–70)  0.605   0.579  0.097        65
            Médio (16–31)   0.541   0.508  0.114        90
            Pequeno (8–12)  0.380   0.343  0.127        95

## 5. Sensibilidade às escolhas de desenho

Três decisões poderiam estar carregando o resultado: a âncora do corte, a
convenção $Mp$ vs. $Mp+1$ e o tratamento de empates.

In [8]:
print("Âncora do corte (médias por ano):")
display(df.groupby('ano_eleicao')[['taa', 'taa_mp1', 'taa_expost_S']].mean().round(3))

Âncora do corte (médias por ano):


,taa,taa_mp1,taa_expost_S
ano_eleicao,,,
2014,0.618,0.534,0.754
2018,0.550,0.428,0.770
2022,0.497,0.408,0.661


- `taa_mp1` — variante Cox M+1, usada em `cap3_necr_decomposicao.py`. Mais baixa,
  como esperado (o $(Mp{+}1)$-ésimo mais financiado se elege menos), sem mudar o
  ordenamento entre anos.
- `taa_expost_S` — corte no nº de cadeiras **efetivamente** conquistadas. Aqui
  $|D| = |E|$, logo precisão = recall = F1. É o teto interpretativo: mesmo
  conhecendo o nº de cadeiras, a mira por recursos acerta ~2/3 a 3/4. Usa
  informação ex-post no tamanho do alvo — robustez, não resultado principal.

In [9]:
# Empates: contagem fracionária vs. desempates arbitrários.
# Se o resultado dependesse da regra de desempate, os três números divergiriam.
from cap3_taa_features import _preparar

prep = _preparar(rrd)
chaves = ['ano_eleicao', 'sg_uf', 'sg_partido_norm']

n_fronteira, linhas = 0, []
for chave, g in prep.groupby(chaves, sort=True):
    mp = int(g['Mp'].iloc[0])
    ncr = int((g['vr_receita_recursos_partidos'] > 0).sum())
    k = min(mp, ncr)
    if mp < 1 or k < 1:
        continue
    rec = g['vr_receita_recursos_partidos'].to_numpy()
    ele = g['eleito'].to_numpy()
    v = np.sort(rec)[::-1]
    empate = k < len(v) and v[k - 1] == v[k]
    n_fronteira += empate

    o_asc = np.lexsort((np.arange(len(rec)), -rec))    # desempate: 1a linha
    o_desc = np.lexsort((-np.arange(len(rec)), -rec))  # desempate: ultima linha
    linhas.append({
        'ano_eleicao': chave[0],
        'fracionario': acertos_fracionarios(rec, ele, k) / k,
        'desempate_A': ele[o_asc][:k].mean(),
        'desempate_B': ele[o_desc][:k].mean(),
    })

emp = pd.DataFrame(linhas)
print(f"listas com empate exato na fronteira do corte: {n_fronteira} de {len(emp)} "
      f"({100 * n_fronteira / len(emp):.1f}%)")
emp.groupby('ano_eleicao')[['fracionario', 'desempate_A', 'desempate_B']].mean().round(4)

listas com empate exato na fronteira do corte: 39 de 770 (5.1%)


,fracionario,desempate_A,desempate_B
ano_eleicao,,,
2014,0.6180,0.6192,0.6165
2018,0.5496,0.5513,0.5493
2022,0.4966,0.5007,0.4894


Empates na fronteira ocorrem em ~5% das listas (partidos repassam valores
redondos idênticos a vários candidatos). A contagem fracionária fica entre os
dois desempates arbitrários e não depende da ordem física das linhas — por isso
é a regra adotada. Note que desempatar por votos, o caminho intuitivo, seria
*ex-post* e inflaria o indicador.

É também a razão para **não** reutilizar `rank_recursos` de `gerar_features`:
ele usa `method="first"`, que desempata pela ordem das linhas do DataFrame.

## 6. Listas extremas

In [10]:
cols = ['ano_eleicao', 'sg_uf', 'sg_partido', 'Mp', 'n_cands',
        'n_com_recursos', 'n_seats', 'k', 'taa', 'base', 'taa_aj']

print("Acerto total (TAA = 1) em listas não-degeneradas, maiores Mp:")
display(df[(df.taa == 1) & ~deg].nlargest(12, 'Mp')[cols])

print("\nErro total (TAA = 0) apesar de o partido ter eleito alguém:")
display(df[(df.taa == 0) & (df.n_seats > 0)].nlargest(12, 'n_seats')[cols])

Acerto total (TAA = 1) em listas não-degeneradas, maiores Mp:


,ano_eleicao,sg_uf,sg_partido,Mp,n_cands,n_com_recursos,n_seats,k,taa,base,taa_aj
609,2022,MG,PL,7,45,44,11,7,1.0,0.244444,1.0
729,2022,RS,PT,5,25,25,6,5,1.0,0.240000,1.0
45,2014,CE,PT,4,10,3,4,3,1.0,0.400000,1.0
60,2014,GO,PMDB,4,11,2,2,2,1.0,0.181818,1.0
121,2014,PE,PSB,4,13,8,8,4,1.0,0.615385,1.0
396,2018,PE,PSB,4,16,13,5,4,1.0,0.312500,1.0
14,2014,AM,PSD,3,5,1,2,1,1.0,0.400000,1.0
23,2014,BA,DEM,3,20,11,4,3,1.0,0.200000,1.0
108,2014,PB,PMDB,3,8,3,3,3,1.0,0.375000,1.0
152,2014,RJ,PP,3,18,15,3,3,1.0,0.166667,1.0



Erro total (TAA = 0) apesar de o partido ter eleito alguém:


,ano_eleicao,sg_uf,sg_partido,Mp,n_cands,n_com_recursos,n_seats,k,taa,base,taa_aj
218,2014,SP,PSC,1,47,1,3,1,0.0,0.063830,-0.068182
421,2018,PR,PSL,1,29,11,3,1,0.0,0.103448,-0.115385
426,2018,RJ,MDB,3,18,18,3,3,0.0,0.166667,-0.200000
565,2022,CE,PSD,2,17,16,3,2,0.0,0.176471,-0.214286
29,2014,BA,PRB,1,6,4,2,1,0.0,0.333333,-0.500000
57,2014,ES,PT,1,9,1,2,1,0.0,0.222222,-0.285714
76,2014,MG,PDT,1,42,38,2,1,0.0,0.047619,-0.050000
352,2018,MS,PSDB,1,5,5,2,1,0.0,0.400000,-0.666667
369,2018,PA,PSDB,1,6,6,2,1,0.0,0.333333,-0.500000
403,2018,PI,PP,1,5,5,2,1,0.0,0.400000,-0.666667


As listas com TAA = 0 **e** cadeiras conquistadas são o caso substantivamente
mais interessante: o partido elegeu deputados, mas nenhum deles estava entre os
$Mp$ que mais financiou. São candidatas naturais a estudo de caso — ou a
alocação foi capturada, ou a expectativa codificada em $Mp$ estava errada.

In [11]:
destino = ROOT / 'data' / 'processed' / 'df_taa_lista.parquet'
df.to_parquet(destino, index=False)
print(f"salvo: {destino} ({len(df)} listas, {df.shape[1]} colunas)")

salvo: C:\Users\yuri.taba\Desktop\recursos-campanha\data\processed\df_taa_lista.parquet (770 listas, 24 colunas)
